In [ ]:
# Step 1: Build a "tech companies" universe from SEC EDGAR
# Output: companies.parquet

import time
import math
import random
import requests
import pandas as pd
import os

# progress bar
try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = lambda x, **kwargs: x

# -------------------------
# Config
# -------------------------
HEADERS = {
    "User-Agent": "CoClaims Research (tommy.zhu@berkeley.edu)"
}

TECH_SIC_RANGES = [
    (3570, 3579),  # Computer & office equipment
    (3660, 3669),  # Communications equipment
    (3670, 3679),  # Electronic components
    (7370, 7379),  # Software & data processing
]

# Common "tech-ish" large caps with sometimes messy SIC classifications
MANUAL_TECH_TICKERS = {
    "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "GOOG", "META", "NFLX", "TSLA",
    "ORCL", "ADBE", "CRM", "INTC", "CSCO", "AVGO", "AMD", "QCOM", "TXN"
}

# Rate limiting and retries
REQUEST_SLEEP_SECONDS = 0.12  # keep you safely below SEC rate controls
MAX_RETRIES = 4
TIMEOUT_SECONDS = 30

# Use None for full run.
LIMIT_COMPANIES = None

# -------------------------
# Helpers
# -------------------------
def _sleep_with_jitter(base_seconds: float):
    time.sleep(base_seconds + random.random() * 0.05)

def get_json_with_retries(url: str, headers: dict, timeout: int = 30, max_retries: int = 4):
    last_exc = None
    for attempt in range(max_retries):
        try:
            r = requests.get(url, headers=headers, timeout=timeout)
            # If SEC throttles, back off
            if r.status_code in (429, 503):
                backoff = (2 ** attempt) * 0.5
                _sleep_with_jitter(backoff)
                continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_exc = e
            backoff = (2 ** attempt) * 0.5
            _sleep_with_jitter(backoff)
    raise RuntimeError(f"Failed after {max_retries} retries for URL: {url}. Last error: {last_exc}")

def is_tech_sic(sic):
    if sic is None or (isinstance(sic, float) and math.isnan(sic)):
        return False
    try:
        s = int(sic)
    except Exception:
        return False
    for low, high in TECH_SIC_RANGES:
        if low <= s <= high:
            return True
    return False

def cik10(cik_int_or_str):
    return str(int(cik_int_or_str)).zfill(10)

# -------------------------
# 1) Load SEC ticker to CIK list
# -------------------------
tickers_url = "https://www.sec.gov/files/company_tickers.json"
tickers_json = get_json_with_retries(
    tickers_url, headers=HEADERS, timeout=TIMEOUT_SECONDS, max_retries=MAX_RETRIES
)

tickers_df = pd.DataFrame.from_dict(tickers_json, orient="index")
tickers_df["cik"] = tickers_df["cik_str"].apply(cik10)

companies_base = tickers_df.rename(columns={"title": "company_name"})[["cik", "ticker", "company_name"]]
companies_base = companies_base.sort_values(["ticker", "cik"]).reset_index(drop=True)

if LIMIT_COMPANIES is not None:
    companies_base = companies_base.head(LIMIT_COMPANIES)

print("Total companies in base universe:", len(companies_base))

# -------------------------
# 2) Enrich each company with SIC from submissions JSON
# -------------------------
def fetch_sic_fields(cik: str):
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    j = get_json_with_retries(url, headers=HEADERS, timeout=TIMEOUT_SECONDS, max_retries=MAX_RETRIES)
    return j.get("sic"), j.get("sicDescription"), j.get("entityType")

rows = []
for _, row in tqdm(companies_base.iterrows(), total=len(companies_base), desc="Fetching SIC from SEC submissions"):
    cik = row["cik"]
    try:
        sic, sic_desc, entity_type = fetch_sic_fields(cik)
    except Exception:
        sic, sic_desc, entity_type = None, None, None

    rows.append({
        "cik": cik,
        "ticker": row["ticker"],
        "company_name": row["company_name"],
        "sic": sic,
        "sic_description": sic_desc,
        "entity_type": entity_type
    })

    _sleep_with_jitter(REQUEST_SLEEP_SECONDS)

companies = pd.DataFrame(rows)

# -------------------------
# 3) Filter to tech SIC ranges
# -------------------------
companies["is_tech_sic"] = companies["sic"].apply(is_tech_sic)
tech_by_sic = companies[companies["is_tech_sic"]].copy()
tech_by_sic["industry_group"] = "tech"

print("Tech companies by SIC:", len(tech_by_sic))

# -------------------------
# 4) Manual whitelist additions
# -------------------------
manual = companies[companies["ticker"].isin(MANUAL_TECH_TICKERS)].copy()
manual["industry_group"] = "tech"
manual["manual_include"] = True

tech_by_sic["manual_include"] = False

tech_companies = pd.concat([tech_by_sic, manual], ignore_index=True)
tech_companies = tech_companies.drop_duplicates(subset=["cik"], keep="first")

# Keep clean columns
tech_companies = tech_companies[[
    "cik",
    "ticker",
    "company_name",
    "sic",
    "sic_description",
    "entity_type",
    "industry_group",
    "manual_include"
]].sort_values(["ticker", "cik"]).reset_index(drop=True)

print("Final tech universe (SIC + manual):", len(tech_companies))

# -------------------------
# 5) Save outputs
# -------------------------
OUTPUT_DIR = "/content/drive/MyDrive/SEC EDGAR Scraper"
os.makedirs(OUTPUT_DIR, exist_ok=True)

tech_companies.to_parquet(f"{OUTPUT_DIR}/companies.parquet", index=False)
tech_companies.to_csv(f"{OUTPUT_DIR}/companies.csv", index=False)

print("Saved: companies.parquet and companies.csv")
tech_companies.head(10)

Total companies in base universe: 10386


Fetching SIC from SEC submissions:   0%|          | 0/10386 [00:00<?, ?it/s]

Tech companies by SIC: 1007
Final tech universe (SIC + manual): 886
Saved: companies.parquet and companies.csv


,cik,ticker,company_name,sic,sic_description,entity_type,industry_group,manual_include
0,0001158114,AAOI,"APPLIED OPTOELECTRONICS, INC.",3674,Semiconductors & Related Devices,operating,tech,False
1,0000320193,AAPL,Apple Inc.,3571,Electronic Computers,operating,tech,False
2,0001971975,ABXXF,Abaxx Technologies Inc.,7372,Services-Prepackaged Software,other,tech,False
3,0000935036,ACIW,"ACI WORLDWIDE, INC.",7372,Services-Prepackaged Software,operating,tech,False
4,0000796343,ADBE,ADOBE INC.,7372,Services-Prepackaged Software,operating,tech,False
5,0002084227,ADBT,"Advasa Holdings, Inc.",7372,Services-Prepackaged Software,other,tech,False
6,0000006281,ADI,ANALOG DEVICES INC,3674,Semiconductors & Related Devices,operating,tech,False
7,0000008670,ADP,AUTOMATIC DATA PROCESSING INC,7374,Services-Computer Processing & Data Preparation,operating,tech,False
8,0000769397,ADSK,"Autodesk, Inc.",7372,Services-Prepackaged Software,operating,tech,False
9,0000926282,ADTN,"ADTRAN Holdings, Inc.",3661,Telephone & Telegraph Apparatus,operating,tech,False


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import time
import random
import requests
import pandas as pd

HEADERS = {"User-Agent": "CoClaims Research (tommy.zhu@berkeley.edu)"}
REQUEST_SLEEP_SECONDS = 0.12
MAX_RETRIES = 4
TIMEOUT_SECONDS = 30

def _sleep_with_jitter(base):
    time.sleep(base + random.random() * 0.05)

def get_json_with_retries(url):
    last_exc = None
    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(url, headers=HEADERS, timeout=TIMEOUT_SECONDS)
            if r.status_code in (429, 503):
                _sleep_with_jitter((2 ** attempt) * 0.5)
                continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_exc = e
            _sleep_with_jitter((2 ** attempt) * 0.5)
    raise RuntimeError(f"Failed: {url}. Last error: {last_exc}")

companies = pd.read_parquet("/content/drive/MyDrive/SEC EDGAR Scraper/companies.parquet")

rows = []
for cik in companies["cik"].tolist():
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    try:
        j = get_json_with_retries(url)
    except Exception:
        _sleep_with_jitter(REQUEST_SLEEP_SECONDS)
        continue

    recent = j.get("filings", {}).get("recent", {})
    forms = recent.get("form", [])
    accession = recent.get("accessionNumber", [])
    filing_dates = recent.get("filingDate", [])
    report_dates = recent.get("reportDate", [])
    primary_docs = recent.get("primaryDocument", [])
    primary_desc = recent.get("primaryDocDescription", [])

    n = len(forms)
    for i in range(n):
        if forms[i] != "10-K":
            continue
        fd = filing_dates[i]
        if fd < "2020-01-01" or fd > "2025-12-31":
            continue

        acc = accession[i]
        acc_nodash = acc.replace("-", "")
        cik_int = str(int(cik))
        base = f"https://www.sec.gov/Archives/edgar/data/{cik_int}/{acc_nodash}/"

        rows.append({
            "cik": cik,
            "accession_number": acc,
            "form_type": forms[i],
            "filing_date": fd,
            "report_date": report_dates[i] if i < len(report_dates) else None,
            "primary_document": primary_docs[i] if i < len(primary_docs) else None,
            "primary_doc_description": primary_desc[i] if i < len(primary_desc) else None,
            "filing_folder_url": base,
            "index_url": base + f"{acc}-index.html",
        })

    _sleep_with_jitter(REQUEST_SLEEP_SECONDS)

filings = pd.DataFrame(rows).drop_duplicates(subset=["cik", "accession_number"])
filings.to_parquet("filings.parquet", index=False)
filings.to_csv("filings.csv", index=False)

print("10-K filings saved:", len(filings))
filings.head(10)

10-K filings saved: 3115


,cik,accession_number,form_type,filing_date,report_date,primary_document,primary_doc_description,filing_folder_url,index_url
0,0001158114,0001437749-25-005575,10-K,2025-02-28,2024-12-31,aaoi20241231_10k.htm,FORM 10-K,https://www.sec.gov/Archives/edgar/data/115811...,https://www.sec.gov/Archives/edgar/data/115811...
1,0001158114,0001437749-24-005367,10-K,2024-02-23,2023-12-31,aaoi20231231_10k.htm,FORM 10-K,https://www.sec.gov/Archives/edgar/data/115811...,https://www.sec.gov/Archives/edgar/data/115811...
2,0001158114,0001437749-23-004648,10-K,2023-02-27,2022-12-31,aaoi20221231_10k.htm,FORM 10-K,https://www.sec.gov/Archives/edgar/data/115811...,https://www.sec.gov/Archives/edgar/data/115811...
3,0001158114,0001437749-22-004349,10-K,2022-02-24,2021-12-31,aaoi20211231_10k.htm,FORM 10-K,https://www.sec.gov/Archives/edgar/data/115811...,https://www.sec.gov/Archives/edgar/data/115811...
4,0001158114,0001437749-21-004108,10-K,2021-02-25,2020-12-31,aaoi20201231_10k.htm,FORM 10-K,https://www.sec.gov/Archives/edgar/data/115811...,https://www.sec.gov/Archives/edgar/data/115811...
5,0001158114,0001437749-20-003909,10-K,2020-02-28,2019-12-31,aaoi20191231_10k.htm,FORM 10-K,https://www.sec.gov/Archives/edgar/data/115811...,https://www.sec.gov/Archives/edgar/data/115811...
6,0000320193,0000320193-25-000079,10-K,2025-10-31,2025-09-27,aapl-20250927.htm,10-K,https://www.sec.gov/Archives/edgar/data/320193...,https://www.sec.gov/Archives/edgar/data/320193...
7,0000320193,0000320193-24-000123,10-K,2024-11-01,2024-09-28,aapl-20240928.htm,10-K,https://www.sec.gov/Archives/edgar/data/320193...,https://www.sec.gov/Archives/edgar/data/320193...
8,0000320193,0000320193-23-000106,10-K,2023-11-03,2023-09-30,aapl-20230930.htm,10-K,https://www.sec.gov/Archives/edgar/data/320193...,https://www.sec.gov/Archives/edgar/data/320193...
9,0000320193,0000320193-22-000108,10-K,2022-10-28,2022-09-24,aapl-20220924.htm,10-K,https://www.sec.gov/Archives/edgar/data/320193...,https://www.sec.gov/Archives/edgar/data/320193...


In [ ]:
import requests
import pandas as pd

HEADERS = {"User-Agent": "CoClaims Research (your.email@domain.com)"}

CONCEPT_WHITELIST = [
    "Revenues",
    "NetIncomeLoss",
    "OperatingIncomeLoss",
    "ResearchAndDevelopmentExpense",
    "Assets",
    "CashAndCashEquivalentsAtCarryingValue",
    "NetCashProvidedByUsedInOperatingActivities",
    "CapitalExpenditures"
]

companies = pd.read_parquet("/content/drive/MyDrive/SEC EDGAR Scraper/companies.parquet")

apple = companies[companies["ticker"] == "AAPL"]
cik = apple.iloc[0]["cik"]

url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
data = requests.get(url, headers=HEADERS).json()

facts = []

us_gaap = data.get("facts", {}).get("us-gaap", {})

for concept in CONCEPT_WHITELIST:
    if concept not in us_gaap:
        continue

    concept_data = us_gaap[concept]
    units = concept_data.get("units", {})

    if "USD" not in units:
        continue

    for item in units["USD"]:
        if item.get("fp") != "FY":
            continue

        fy = item.get("fy")
        if fy is None or fy < 2020 or fy > 2025:
            continue

        facts.append({
            "cik": cik,
            "ticker": "AAPL",
            "concept": concept,
            "value": item.get("val"),
            "fy": fy,
            "form": item.get("form"),
            "filed": item.get("filed"),
            "end_date": item.get("end"),
            "accession_number": item.get("accn")
        })

apple_facts = pd.DataFrame(facts)
apple_facts.sort_values(["concept", "fy"]).head(20)

,cik,ticker,concept,value,fy,form,filed,end_date,accession_number
62,0000320193,AAPL,Assets,338516000000,2020,10-K,2020-10-30,2019-09-28,0000320193-20-000096
63,0000320193,AAPL,Assets,323888000000,2020,10-K,2020-10-30,2020-09-26,0000320193-20-000096
64,0000320193,AAPL,Assets,323888000000,2021,10-K,2021-10-29,2020-09-26,0000320193-21-000105
65,0000320193,AAPL,Assets,351002000000,2021,10-K,2021-10-29,2021-09-25,0000320193-21-000105
66,0000320193,AAPL,Assets,351002000000,2022,10-K,2022-10-28,2021-09-25,0000320193-22-000108
67,0000320193,AAPL,Assets,352755000000,2022,10-K,2022-10-28,2022-09-24,0000320193-22-000108
68,0000320193,AAPL,Assets,352755000000,2023,10-K,2023-11-03,2022-09-24,0000320193-23-000106
69,0000320193,AAPL,Assets,352583000000,2023,10-K,2023-11-03,2023-09-30,0000320193-23-000106
70,0000320193,AAPL,Assets,352583000000,2024,10-K,2024-11-01,2023-09-30,0000320193-24-000123
71,0000320193,AAPL,Assets,364980000000,2024,10-K,2024-11-01,2024-09-28,0000320193-24-000123


In [ ]:
apple_facts["end_date"] = pd.to_datetime(apple_facts["end_date"])

apple_facts = (
    apple_facts
    .sort_values("end_date")
    .groupby(["cik", "concept", "fy"], as_index=False)
    .last()
)

apple_facts.sort_values(["concept", "fy"]).head(20)

,cik,concept,fy,ticker,value,form,filed,end_date,accession_number
0,0000320193,Assets,2020,AAPL,323888000000,10-K,2020-10-30,2020-09-26,0000320193-20-000096
1,0000320193,Assets,2021,AAPL,351002000000,10-K,2021-10-29,2021-09-25,0000320193-21-000105
2,0000320193,Assets,2022,AAPL,352755000000,10-K,2022-10-28,2022-09-24,0000320193-22-000108
3,0000320193,Assets,2023,AAPL,352583000000,10-K,2023-11-03,2023-09-30,0000320193-23-000106
4,0000320193,Assets,2024,AAPL,364980000000,10-K,2024-11-01,2024-09-28,0000320193-24-000123
5,0000320193,Assets,2025,AAPL,359241000000,10-K,2025-10-31,2025-09-27,0000320193-25-000079
6,0000320193,CashAndCashEquivalentsAtCarryingValue,2020,AAPL,38016000000,10-K,2020-10-30,2020-09-26,0000320193-20-000096
7,0000320193,CashAndCashEquivalentsAtCarryingValue,2021,AAPL,34940000000,10-K,2021-10-29,2021-09-25,0000320193-21-000105
8,0000320193,CashAndCashEquivalentsAtCarryingValue,2022,AAPL,23646000000,10-K,2022-10-28,2022-09-24,0000320193-22-000108
9,0000320193,CashAndCashEquivalentsAtCarryingValue,2023,AAPL,29965000000,10-K,2023-11-03,2023-09-30,0000320193-23-000106


In [ ]:
import time
import random
from tqdm.auto import tqdm

HEADERS = {"User-Agent": "CoClaims Research (tommy.zhu@berkeley.edu)"}
REQUEST_SLEEP_SECONDS = 0.12

companies = pd.read_parquet("/content/drive/MyDrive/SEC EDGAR Scraper/companies.parquet")

CONCEPT_WHITELIST = [
    "Revenues",
    "NetIncomeLoss",
    "OperatingIncomeLoss",
    "ResearchAndDevelopmentExpense",
    "Assets",
    "CashAndCashEquivalentsAtCarryingValue",
    "NetCashProvidedByUsedInOperatingActivities",
    "CapitalExpenditures"
]

all_facts = []

for _, row in tqdm(companies.iterrows(), total=len(companies), desc="Pulling XBRL facts"):
    cik = row["cik"]
    ticker = row["ticker"]

    try:
        url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
        data = requests.get(url, headers=HEADERS, timeout=30).json()
    except:
        continue

    us_gaap = data.get("facts", {}).get("us-gaap", {})

    for concept in CONCEPT_WHITELIST:
        if concept not in us_gaap:
            continue

        units = us_gaap[concept].get("units", {})
        if "USD" not in units:
            continue

        for item in units["USD"]:
            if item.get("fp") != "FY":
                continue

            fy = item.get("fy")
            if fy is None or fy < 2020 or fy > 2025:
                continue

            all_facts.append({
                "cik": cik,
                "ticker": ticker,
                "concept": concept,
                "value": item.get("val"),
                "fy": fy,
                "form": item.get("form"),
                "filed": item.get("filed"),
                "end_date": item.get("end"),
                "accession_number": item.get("accn")
            })

    time.sleep(REQUEST_SLEEP_SECONDS)

facts_df = pd.DataFrame(all_facts)

OUTPUT_DIR = "/content/drive/MyDrive/SEC EDGAR Scraper"
facts_df.to_parquet(f"{OUTPUT_DIR}/facts.parquet", index=False)
facts_df.to_csv(f"{OUTPUT_DIR}/facts.csv", index=False)

print("Facts saved:", len(facts_df))
facts_df.head()

Pulling XBRL facts:   0%|          | 0/889 [00:00<?, ?it/s]

Facts saved: 58432


,cik,ticker,concept,value,fy,form,filed,end_date,accession_number
0,0001158114,AAOI,NetIncomeLoss,-2146000.0,2020,10-K,2021-02-25,2018-12-31,0001437749-21-004108
1,0001158114,AAOI,NetIncomeLoss,-10474000.0,2020,10-K,2021-02-25,2019-03-31,0001437749-21-004108
2,0001158114,AAOI,NetIncomeLoss,-11366000.0,2020,10-K,2021-02-25,2019-06-30,0001437749-21-004108
3,0001158114,AAOI,NetIncomeLoss,-8780000.0,2020,10-K,2021-02-25,2019-09-30,0001437749-21-004108
4,0001158114,AAOI,NetIncomeLoss,-66049000.0,2020,10-K,2021-02-25,2019-12-31,0001437749-21-004108


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=facts_df)

https://docs.google.com/spreadsheets/d/1IgOwBH0uXEepSTc0mjW6mT0ZG3S1phqflwUxBY5ADzI/edit#gid=0


In [ ]:
# A -> Small B -> Scale: Extract MD&A sections into mdna_sections.parquet
# Prereqs: filings.parquet exists in your Drive folder
# Installs (once): !pip install -q beautifulsoup4 lxml pyarrow tqdm

import os, re, time, random, requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

HEADERS = {"User-Agent": "CoClaims Research (your.email@domain.com)"}  # replace
BASE_DIR = "/content/drive/MyDrive/SEC EDGAR Scraper"
FILINGS_PATH = f"{BASE_DIR}/filings.parquet"
OUT_PATH = f"{BASE_DIR}/mdna_sections.parquet"
REQUEST_SLEEP_SECONDS = 0.15

filings = pd.read_parquet(FILINGS_PATH)

def clean_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def html_to_text(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["script", "style", "noscript", "table"]):
        tag.decompose()
    text = soup.get_text(" ")
    return clean_text(text)

def find_span(text: str, start_patterns, end_patterns):
    # Returns (start_idx, end_idx) in text, or (None, None)
    start_idx = None
    for pat in start_patterns:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            start_idx = m.start()
            break
    if start_idx is None:
        return None, None

    end_idx = None
    tail = text[start_idx:]
    for pat in end_patterns:
        m = re.search(pat, tail, flags=re.IGNORECASE)
        if m:
            end_idx = start_idx + m.start()
            break

    if end_idx is None:
        end_idx = min(len(text), start_idx + 120_000)  # safe cutoff
    return start_idx, end_idx

def extract_mdna(text: str, form_type: str):
    # Heuristics:
    # 10-K: Item 7 -> Item 7A or Item 8
    # 10-Q: Item 2 -> Item 3
    if form_type == "10-K":
        start_patterns = [
            r"\bitem\s*7\.\s*management[’']?s\s+discussion\s+and\s+analysis\b",
            r"\bitem\s*7\.\s*management\s+discussion\s+and\s+analysis\b",
        ]
        end_patterns = [
            r"\bitem\s*7a\.\b",
            r"\bitem\s*8\.\s*financial\s+statements\b",
            r"\bitem\s*8\.\b",
        ]
    else:  # 10-Q
        start_patterns = [
            r"\bitem\s*2\.\s*management[’']?s\s+discussion\s+and\s+analysis\b",
            r"\bitem\s*2\.\s*management\s+discussion\s+and\s+analysis\b",
        ]
        end_patterns = [
            r"\bitem\s*3\.\s*quantitative\b",
            r"\bitem\s*3\.\b",
        ]

    s, e = find_span(text, start_patterns, end_patterns)
    if s is None:
        return None
    mdna = text[s:e]
    if len(mdna) < 1500:  # too short, likely mis-detected
        return None
    return mdna

def fetch_html(url: str) -> str | None:
    try:
        r = requests.get(url, headers=HEADERS, timeout=40)
        if r.status_code != 200:
            return None
        return r.text
    except Exception:
        return None

# --------
# B) Small dry run first
# --------
TEST_TICKERS = {"AAPL", "MSFT", "NVDA"}  # adjust
test_filings = filings.merge(
    pd.read_parquet(f"{BASE_DIR}/companies.parquet")[["cik","ticker"]],
    on="cik",
    how="left"
)
test_filings = test_filings[(test_filings["ticker"].isin(TEST_TICKERS)) & (test_filings["form_type"].isin(["10-K"]))].copy()
test_filings = test_filings.sort_values(["ticker", "filing_date"])

rows = []
for _, r in tqdm(test_filings.iterrows(), total=len(test_filings), desc="Dry run MD&A"):
    filing_url = r["filing_folder_url"] + (r["primary_document"] or "")
    html = fetch_html(filing_url)
    if not html:
        time.sleep(REQUEST_SLEEP_SECONDS)
        continue

    text = html_to_text(html)
    mdna = extract_mdna(text, r["form_type"])
    if mdna:
        rows.append({
            "cik": r["cik"],
            "ticker": r.get("ticker"),
            "accession_number": r["accession_number"],
            "form_type": r["form_type"],
            "filing_date": r["filing_date"],
            "report_date": r.get("report_date"),
            "section": "MD&A",
            "source_url": filing_url,
            "mdna_text": mdna,
            "mdna_chars": len(mdna),
        })

    time.sleep(REQUEST_SLEEP_SECONDS + random.random() * 0.05)

mdna_test = pd.DataFrame(rows)
print("Dry run rows:", len(mdna_test))
mdna_test[["ticker","filing_date","mdna_chars","source_url"]].head(10)

Dry run MD&A:   0%|          | 0/18 [00:00<?, ?it/s]

/tmp/ipython-input-1549205951.py:23: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


Dry run rows: 18


,ticker,filing_date,mdna_chars,source_url
0,AAPL,2020-10-30,36015,https://www.sec.gov/Archives/edgar/data/320193...
1,AAPL,2021-10-29,23756,https://www.sec.gov/Archives/edgar/data/320193...
2,AAPL,2022-10-28,22574,https://www.sec.gov/Archives/edgar/data/320193...
3,AAPL,2023-11-03,16748,https://www.sec.gov/Archives/edgar/data/320193...
4,AAPL,2024-11-01,16680,https://www.sec.gov/Archives/edgar/data/320193...
5,AAPL,2025-10-31,19364,https://www.sec.gov/Archives/edgar/data/320193...
6,MSFT,2020-07-30,49587,https://www.sec.gov/Archives/edgar/data/789019...
7,MSFT,2021-07-29,44830,https://www.sec.gov/Archives/edgar/data/789019...
8,MSFT,2022-07-28,44509,https://www.sec.gov/Archives/edgar/data/789019...
9,MSFT,2023-07-27,120000,https://www.sec.gov/Archives/edgar/data/789019...


In [ ]:
print(mdna_test.shape)
mdna_test.columns

(18, 10)


Index(['cik', 'ticker', 'accession_number', 'form_type', 'filing_date',
       'report_date', 'section', 'source_url', 'mdna_text', 'mdna_chars'],
      dtype='object')

In [ ]:
keywords = ["demand", "pricing", "margin", "inflation", "supply", "guidance"]

for kw in keywords:
    print(kw, sample.lower().count(kw))

demand 8
pricing 11
margin 21
inflation 0
supply 0
guidance 6


In [ ]:
# Phase B (v1): Extract abstract MD&A facts (Demand, Pricing, Margin) from mdna_test
# Output: mdna_facts_v1.parquet + mdna_facts_v1.csv in your Drive folder
# Assumes you already have: mdna_test DataFrame with columns incl. mdna_text, ticker, cik, accession_number, form_type, filing_date, source_url

import re
import pandas as pd

OUT_DIR = "/content/drive/MyDrive/SEC EDGAR Scraper"

# -------------------------
# 1) Sentence splitting (works even if whitespace was flattened)
# -------------------------
def split_sentences(text: str):
    sents = re.split(r'(?<=[\.\?\!])\s+(?=[A-Z])', text)
    sents = [re.sub(r"\s+", " ", s).strip() for s in sents]
    return [s for s in sents if 80 <= len(s) <= 900]

# -------------------------
# 2) Noise filter (removes table-ish / header-ish artifacts)
# -------------------------
def is_noise(s: str) -> bool:
    s_l = s.lower()
    if len(s) < 100:
        return True
    if "form 10-k" in s_l or "form 10-q" in s_l:
        return True
    if "table shows" in s_l:
        return True
    if s.count("|") > 2:
        return True
    if "(dollars in millions)" in s_l:
        return True
    # very number-dense lines tend to be table remnants
    if len(re.findall(r"\d", s)) > 80:
        return True
    return False

# -------------------------
# 3) Fact detectors (tighter Demand to avoid FX false positives)
# -------------------------
PRICING_PATTERNS = [
    r"\bpricing\b",
    r"\bprice(s)?\b",
    r"\baverage\s+selling\s+price(s)?\b|\bASP(s)?\b",
    r"\bdiscount(s|ing)?\b",
    r"\bpromot(ion|ional)\b",
    r"\bprice\s+pressure\b|\bcompetitive\s+pricing\b",
]

def is_pricing_sentence(s: str) -> bool:
    return any(re.search(p, s, flags=re.IGNORECASE) for p in PRICING_PATTERNS)

def is_demand_sentence(s: str) -> bool:
    s_l = s.lower()
    # demand explicitly
    if "demand" in s_l:
        return True
    # orders/bookings as demand proxies, but avoid FX/currency context
    if ("orders" in s_l or "bookings" in s_l) and ("currency" not in s_l and "foreign exchange" not in s_l and "fx" not in s_l):
        return True
    return False

def is_margin_driver(s: str) -> bool:
    s_l = s.lower()
    # margin + driver language
    if "margin" in s_l and ("due to" in s_l or "primarily" in s_l or "driven by" in s_l or "attributable to" in s_l):
        return True
    # gross margin specifically
    if "gross margin" in s_l and ("increase" in s_l or "decrease" in s_l or "improve" in s_l or "decline" in s_l):
        return True
    return False

# -------------------------
# 4) Direction classifier (simple cue counting)
# -------------------------
NEG_CUES = r"\b(decrease|decline|down|lower|pressure|headwind|soft|weak|challenging|unfavorable|adverse|constraint|shortage)\b"
POS_CUES = r"\b(increase|grow|up|higher|strong|improve|favorable|benefit|tailwind|expand|expansion|accelerate|record)\b"

def classify_direction(s: str) -> str:
    s_l = s.lower()
    neg = len(re.findall(NEG_CUES, s_l))
    pos = len(re.findall(POS_CUES, s_l))
    if neg > pos and neg >= 1:
        return "negative"
    if pos > neg and pos >= 1:
        return "positive"
    return "neutral"

# -------------------------
# 5) Extract facts
# -------------------------
facts = []

for _, r in mdna_test.iterrows():
    sents = split_sentences(r["mdna_text"])

    for i, s in enumerate(sents):
        if is_noise(s):
            continue

        fact_type = None
        # Priority order (to keep 1 label per sentence)
        if is_margin_driver(s):
            fact_type = "MARGIN_DRIVER"
        elif is_pricing_sentence(s):
            fact_type = "PRICING_PRESSURE"
        elif is_demand_sentence(s):
            fact_type = "DEMAND_TREND"
        else:
            continue

        facts.append({
            "cik": r["cik"],
            "ticker": r["ticker"],
            "accession_number": r["accession_number"],
            "form_type": r["form_type"],
            "filing_date": r["filing_date"],
            "section": "MD&A",
            "fact_type": fact_type,
            "direction": classify_direction(s),
            "evidence_text": s,
            "source_url": r["source_url"],
            "sent_index": i,
        })

mdna_facts_v0 = pd.DataFrame(
    facts,
    columns=[
        "cik","ticker","accession_number","form_type","filing_date","section",
        "fact_type","direction","evidence_text","source_url","sent_index"
    ]
)

print("Extracted candidate facts (pre-dedup):", len(mdna_facts_v0))
print(mdna_facts_v0["fact_type"].value_counts() if len(mdna_facts_v0) else mdna_facts_v0)

# -------------------------
# 6) Deduplicate (same sentence repeated)
# -------------------------
if len(mdna_facts_v0):
    mdna_facts_v0["evidence_norm"] = (
        mdna_facts_v0["evidence_text"]
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    mdna_facts_v1 = (
        mdna_facts_v0
        .drop_duplicates(subset=["ticker","filing_date","fact_type","evidence_norm"])
        .drop(columns=["evidence_norm"])
        .reset_index(drop=True)
    )
else:
    mdna_facts_v1 = mdna_facts_v0.copy()

print("After dedup:", len(mdna_facts_v1))
print(mdna_facts_v1["fact_type"].value_counts() if len(mdna_facts_v1) else mdna_facts_v1)

# -------------------------
# 7) Save to Drive
# -------------------------
mdna_facts_v1.to_parquet(f"{OUT_DIR}/mdna_facts_v1.parquet", index=False)
mdna_facts_v1.to_csv(f"{OUT_DIR}/mdna_facts_v1.csv", index=False)

print("Saved:", f"{OUT_DIR}/mdna_facts_v1.parquet")
mdna_facts_v1.head(20)

Extracted candidate facts (pre-dedup): 447
fact_type
PRICING_PRESSURE    239
DEMAND_TREND        139
MARGIN_DRIVER        69
Name: count, dtype: int64
After dedup: 442
fact_type
PRICING_PRESSURE    235
DEMAND_TREND        138
MARGIN_DRIVER        69
Name: count, dtype: int64
Saved: /content/drive/MyDrive/SEC EDGAR Scraper/mdna_facts_v1.parquet


,cik,ticker,accession_number,form_type,filing_date,section,fact_type,direction,evidence_text,source_url,sent_index
0,0000320193,AAPL,0000320193-20-000096,10-K,2020-10-30,MD&A,DEMAND_TREND,neutral,Such measures have included restrictions on tr...,https://www.sec.gov/Archives/edgar/data/320193...,4
1,0000320193,AAPL,0000320193-20-000096,10-K,2020-10-30,MD&A,PRICING_PRESSURE,neutral,The COVID-19 pandemic and the measures taken b...,https://www.sec.gov/Archives/edgar/data/320193...,6
2,0000320193,AAPL,0000320193-20-000096,10-K,2020-10-30,MD&A,DEMAND_TREND,neutral,The full extent of the future impact of the CO...,https://www.sec.gov/Archives/edgar/data/320193...,9
3,0000320193,AAPL,0000320193-20-000096,10-K,2020-10-30,MD&A,PRICING_PRESSURE,neutral,Services net sales also include amortization o...,https://www.sec.gov/Archives/edgar/data/320193...,19
4,0000320193,AAPL,0000320193-20-000096,10-K,2020-10-30,MD&A,MARGIN_DRIVER,positive,Products gross margin percentage decreased dur...,https://www.sec.gov/Archives/edgar/data/320193...,39
5,0000320193,AAPL,0000320193-20-000096,10-K,2020-10-30,MD&A,MARGIN_DRIVER,positive,Services Gross Margin Services gross margin in...,https://www.sec.gov/Archives/edgar/data/320193...,40
6,0000320193,AAPL,0000320193-20-000096,10-K,2020-10-30,MD&A,MARGIN_DRIVER,positive,Services gross margin percentage increased dur...,https://www.sec.gov/Archives/edgar/data/320193...,41
7,0000320193,AAPL,0000320193-20-000096,10-K,2020-10-30,MD&A,DEMAND_TREND,positive,These outsourcing partners acquire components ...,https://www.sec.gov/Archives/edgar/data/320193...,87
8,0000320193,AAPL,0000320193-20-000096,10-K,2020-10-30,MD&A,PRICING_PRESSURE,neutral,"The first performance obligation, which repres...",https://www.sec.gov/Archives/edgar/data/320193...,101
9,0000320193,AAPL,0000320193-20-000096,10-K,2020-10-30,MD&A,PRICING_PRESSURE,neutral,The Company allocates revenue and any related ...,https://www.sec.gov/Archives/edgar/data/320193...,104


In [ ]:
# FULL MD&A DATASET (Scale) + ABSTRACT FACT EXTRACTION
# Produces:
# 1) mdna_sections.parquet  (all 10-K MD&A sections, 2020–2025)
# 2) mdna_facts_v1.parquet  (Demand, Pricing, Margin, Strategic Investment Focus)
#
# Prereqs in Drive folder:
# - companies.parquet
# - filings.parquet
#
# Notes:
# - This pipeline is designed for Colab + Drive persistence.
# - Uses sentence splitting because earlier text cleaning may flatten paragraphs.

import os, re, time, random, requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

BASE_DIR = "/content/drive/MyDrive/SEC EDGAR Scraper"
os.makedirs(BASE_DIR, exist_ok=True)

COMPANIES_PATH = f"{BASE_DIR}/companies.parquet"
FILINGS_PATH   = f"{BASE_DIR}/filings.parquet"

MDNA_SECTIONS_PATH = f"{BASE_DIR}/mdna_sections.parquet"
MDNA_SECTIONS_CSV  = f"{BASE_DIR}/mdna_sections.csv"

MDNA_FACTS_PATH = f"{BASE_DIR}/mdna_facts_v1.parquet"
MDNA_FACTS_CSV  = f"{BASE_DIR}/mdna_facts_v1.csv"

HEADERS = {"User-Agent": "CoClaims Research (your.email@domain.com)"}  # replace
REQUEST_SLEEP_SECONDS = 0.15

# -------------------------
# A) Build mdna_sections.parquet (10-K only, 2020–2025)
# -------------------------
def fetch_html(url: str) -> str | None:
    try:
        r = requests.get(url, headers=HEADERS, timeout=40)
        if r.status_code != 200:
            return None
        return r.text
    except Exception:
        return None

def html_to_text_keep_breaks(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")

    # Remove heavy noise (tables are the biggest offender)
    for tag in soup(["script", "style", "noscript", "table"]):
        tag.decompose()

    # Preserve some structure
    for br in soup.find_all("br"):
        br.replace_with("\n")
    for tag in soup.find_all(["p", "div", "li", "h1", "h2", "h3", "h4"]):
        tag.insert_before("\n")
        tag.insert_after("\n")

    text = soup.get_text()
    text = re.sub(r"[ \t\r\f\v]+", " ", text)   # normalize whitespace (not \n)
    text = re.sub(r"\n{3,}", "\n\n", text)      # compress multiple newlines
    return text.strip()

def find_span(text: str, start_patterns, end_patterns):
    start_idx = None
    for pat in start_patterns:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            start_idx = m.start()
            break
    if start_idx is None:
        return None, None

    end_idx = None
    tail = text[start_idx:]
    for pat in end_patterns:
        m = re.search(pat, tail, flags=re.IGNORECASE)
        if m:
            end_idx = start_idx + m.start()
            break

    if end_idx is None:
        end_idx = min(len(text), start_idx + 250_000)  # safety cutoff
    return start_idx, end_idx

def extract_mdna(text: str, form_type: str) -> str | None:
    if form_type != "10-K":
        return None

    start_patterns = [
        r"\bitem\s*7\.\s*management[’']?s\s+discussion\s+and\s+analysis\b",
        r"\bitem\s*7\.\s*management\s+discussion\s+and\s+analysis\b",
    ]
    end_patterns = [
        r"\bitem\s*7a\.\b",
        r"\bitem\s*8\.\s*financial\s+statements\b",
        r"\bitem\s*8\.\b",
    ]

    s, e = find_span(text, start_patterns, end_patterns)
    if s is None:
        return None

    mdna = text[s:e].strip()
    if len(mdna) < 2000:
        return None
    return mdna

def build_mdna_sections():
    companies = pd.read_parquet(COMPANIES_PATH)[["cik", "ticker"]]
    filings = pd.read_parquet(FILINGS_PATH)

    df = filings.merge(companies, on="cik", how="left")
    df = df[df["form_type"] == "10-K"].copy()

    # Filings date filter 2020–2025 (string compare works for YYYY-MM-DD)
    df = df[(df["filing_date"] >= "2020-01-01") & (df["filing_date"] <= "2025-12-31")].copy()
    df = df.sort_values(["ticker", "filing_date"])

    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc="Building mdna_sections"):
        primary_doc = r.get("primary_document")
        if not isinstance(primary_doc, str) or primary_doc.strip() == "":
            time.sleep(REQUEST_SLEEP_SECONDS)
            continue

        filing_url = r["filing_folder_url"] + primary_doc
        html = fetch_html(filing_url)
        if not html:
            time.sleep(REQUEST_SLEEP_SECONDS)
            continue

        text = html_to_text_keep_breaks(html)
        mdna = extract_mdna(text, r["form_type"])
        if mdna:
            rows.append({
                "cik": r["cik"],
                "ticker": r["ticker"],
                "accession_number": r["accession_number"],
                "form_type": r["form_type"],
                "filing_date": r["filing_date"],
                "report_date": r.get("report_date"),
                "section": "MD&A",
                "source_url": filing_url,
                "mdna_text": mdna,
                "mdna_chars": len(mdna),
            })

        time.sleep(REQUEST_SLEEP_SECONDS + random.random() * 0.05)

    mdna_sections = pd.DataFrame(rows)
    mdna_sections.to_parquet(MDNA_SECTIONS_PATH, index=False)
    mdna_sections.to_csv(MDNA_SECTIONS_CSV, index=False)
    print("Saved mdna_sections:", len(mdna_sections))
    return mdna_sections

# -------------------------
# B) Extract abstract facts from mdna_sections (Demand, Pricing, Margin, Strategic Investment Focus)
# -------------------------
def split_sentences(text: str):
    sents = re.split(r'(?<=[\.\?\!])\s+(?=[A-Z])', text)
    sents = [re.sub(r"\s+", " ", s).strip() for s in sents]
    return [s for s in sents if 80 <= len(s) <= 900]

def is_noise(s: str) -> bool:
    s_l = s.lower()
    if len(s) < 100:
        return True
    if "form 10-k" in s_l:
        return True
    if "table shows" in s_l:
        return True
    if s.count("|") > 2:
        return True
    if "(dollars in millions)" in s_l:
        return True
    if len(re.findall(r"\d", s)) > 80:
        return True
    return False

PRICING_PATTERNS = [
    r"\bpricing\b",
    r"\bprice(s)?\b",
    r"\baverage\s+selling\s+price(s)?\b|\bASP(s)?\b",
    r"\bdiscount(s|ing)?\b",
    r"\bpromot(ion|ional)\b",
    r"\bprice\s+pressure\b|\bcompetitive\s+pricing\b",
]

def is_pricing_sentence(s: str) -> bool:
    return any(re.search(p, s, flags=re.IGNORECASE) for p in PRICING_PATTERNS)

def is_demand_sentence(s: str) -> bool:
    s_l = s.lower()
    if "demand" in s_l:
        return True
    if ("orders" in s_l or "bookings" in s_l) and ("currency" not in s_l and "foreign exchange" not in s_l and "fx" not in s_l):
        return True
    return False

def is_margin_driver(s: str) -> bool:
    s_l = s.lower()
    if "margin" in s_l and ("due to" in s_l or "primarily" in s_l or "driven by" in s_l or "attributable to" in s_l):
        return True
    if "gross margin" in s_l and ("increase" in s_l or "decrease" in s_l or "improve" in s_l or "decline" in s_l):
        return True
    return False

STRATEGIC_INVESTMENT_PATTERNS = [
    r"\binvest(ing|ment)\b.*\b(ai|artificial intelligence|cloud|data center|platform)\b",
    r"\bfocus(ed)?\b.*\b(ai|artificial intelligence|cloud|data center|platform)\b",
    r"\bstrategic\b.*\binitiative(s)?\b",
    r"\btransformation\b|\bpivot\b|\broadmap\b",
    r"\bcapabilit(y|ies)\b.*\b(ai|cloud)\b",
]

def is_strategic_investment_focus(s: str) -> bool:
    return any(re.search(p, s, flags=re.IGNORECASE) for p in STRATEGIC_INVESTMENT_PATTERNS)

NEG_CUES = r"\b(decrease|decline|down|lower|pressure|headwind|soft|weak|challenging|unfavorable|adverse|constraint|shortage)\b"
POS_CUES = r"\b(increase|grow|up|higher|strong|improve|favorable|benefit|tailwind|expand|expansion|accelerate|record)\b"

def classify_direction(s: str) -> str:
    s_l = s.lower()
    neg = len(re.findall(NEG_CUES, s_l))
    pos = len(re.findall(POS_CUES, s_l))
    if neg > pos and neg >= 1:
        return "negative"
    if pos > neg and pos >= 1:
        return "positive"
    return "neutral"

def extract_mdna_facts(mdna_sections: pd.DataFrame) -> pd.DataFrame:
    facts = []

    for _, r in tqdm(mdna_sections.iterrows(), total=len(mdna_sections), desc="Extracting abstract MD&A facts"):
        sents = split_sentences(r["mdna_text"])

        for i, s in enumerate(sents):
            if is_noise(s):
                continue

            # Priority order: more specific first
            fact_type = None
            if is_strategic_investment_focus(s):
                fact_type = "STRATEGIC_INVESTMENT_FOCUS"
            elif is_margin_driver(s):
                fact_type = "MARGIN_DRIVER"
            elif is_pricing_sentence(s):
                fact_type = "PRICING_PRESSURE"
            elif is_demand_sentence(s):
                fact_type = "DEMAND_TREND"
            else:
                continue

            facts.append({
                "cik": r["cik"],
                "ticker": r["ticker"],
                "accession_number": r["accession_number"],
                "form_type": r["form_type"],
                "filing_date": r["filing_date"],
                "section": "MD&A",
                "fact_type": fact_type,
                "direction": classify_direction(s),
                "evidence_text": s,
                "source_url": r["source_url"],
                "sent_index": i,
            })

    facts_df = pd.DataFrame(
        facts,
        columns=[
            "cik","ticker","accession_number","form_type","filing_date","section",
            "fact_type","direction","evidence_text","source_url","sent_index"
        ]
    )

    # Dedup exact repeats
    if len(facts_df):
        facts_df["evidence_norm"] = (
            facts_df["evidence_text"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
        )
        facts_df = (
            facts_df
            .drop_duplicates(subset=["ticker","filing_date","fact_type","evidence_norm"])
            .drop(columns=["evidence_norm"])
            .reset_index(drop=True)
        )

    return facts_df

# -------------------------
# RUN
# -------------------------
# Build full MD&A sections (comment out if you already built mdna_sections.parquet)
mdna_sections = build_mdna_sections()

# Or load if already built:
# mdna_sections = pd.read_parquet(MDNA_SECTIONS_PATH)

mdna_facts_v1 = extract_mdna_facts(mdna_sections)

mdna_facts_v1.to_parquet(MDNA_FACTS_PATH, index=False)
mdna_facts_v1.to_csv(MDNA_FACTS_CSV, index=False)

print("Saved mdna_facts_v1:", len(mdna_facts_v1))
print(mdna_facts_v1["fact_type"].value_counts())
mdna_facts_v1.head(20)

Building mdna_sections:   0%|          | 0/3115 [00:00<?, ?it/s]

/tmp/ipython-input-799/1005943702.py:47: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


Saved mdna_sections: 2553


Extracting abstract MD&A facts:   0%|          | 0/2553 [00:00<?, ?it/s]

Saved mdna_facts_v1: 66080
fact_type
PRICING_PRESSURE              42336
DEMAND_TREND                  15409
MARGIN_DRIVER                  4817
STRATEGIC_INVESTMENT_FOCUS     3518
Name: count, dtype: int64


,cik,ticker,accession_number,form_type,filing_date,section,fact_type,direction,evidence_text,source_url,sent_index
0,0000320193,AAPL,0000320193-20-000096,10-K,2020-10-30,MD&A,DEMAND_TREND,neutral,Such measures have included restrictions on tr...,https://www.sec.gov/Archives/edgar/data/320193...,4
1,0000320193,AAPL,0000320193-20-000096,10-K,2020-10-30,MD&A,PRICING_PRESSURE,neutral,The COVID-19 pandemic and the measures taken b...,https://www.sec.gov/Archives/edgar/data/320193...,6
2,0000320193,AAPL,0000320193-20-000096,10-K,2020-10-30,MD&A,DEMAND_TREND,neutral,The full extent of the future impact of the CO...,https://www.sec.gov/Archives/edgar/data/320193...,9
3,0000320193,AAPL,0000320193-20-000096,10-K,2020-10-30,MD&A,PRICING_PRESSURE,neutral,Services net sales also include amortization o...,https://www.sec.gov/Archives/edgar/data/320193...,19
4,0000320193,AAPL,0000320193-20-000096,10-K,2020-10-30,MD&A,MARGIN_DRIVER,positive,Products gross margin percentage decreased dur...,https://www.sec.gov/Archives/edgar/data/320193...,39
5,0000320193,AAPL,0000320193-20-000096,10-K,2020-10-30,MD&A,MARGIN_DRIVER,positive,Services Gross Margin Services gross margin in...,https://www.sec.gov/Archives/edgar/data/320193...,40
6,0000320193,AAPL,0000320193-20-000096,10-K,2020-10-30,MD&A,MARGIN_DRIVER,positive,Services gross margin percentage increased dur...,https://www.sec.gov/Archives/edgar/data/320193...,41
7,0000320193,AAPL,0000320193-20-000096,10-K,2020-10-30,MD&A,DEMAND_TREND,positive,These outsourcing partners acquire components ...,https://www.sec.gov/Archives/edgar/data/320193...,87
8,0000320193,AAPL,0000320193-20-000096,10-K,2020-10-30,MD&A,PRICING_PRESSURE,neutral,"The first performance obligation, which repres...",https://www.sec.gov/Archives/edgar/data/320193...,101
9,0000320193,AAPL,0000320193-20-000096,10-K,2020-10-30,MD&A,PRICING_PRESSURE,neutral,The Company allocates revenue and any related ...,https://www.sec.gov/Archives/edgar/data/320193...,104
